# 38 — LangGraph Command Pattern

LangGraph's `Command` primitive lets a node return routing decisions and state updates in a single object — no `add_conditional_edges` required. The router classifies the task and returns `Command(goto="code"|"explain"|"math"|"creative", update={...})`.

**What you'll learn**
- `Command(goto=node_name, update={...})` — routing + state update in one return
- How to skip `add_conditional_edges` entirely
- 4-specialist pattern: one router, four independent handlers
- Why co-locating routing logic inside the node improves readability
- Fallback routing: handle unrecognized categories gracefully

**Contrast:** `add_conditional_edges` separates routing into a mapping dict. `Command` keeps the routing decision co-located with the logic that makes it.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()
if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Command vs conditional_edges ──────────────────────────────────────────
#
# conditional_edges style:
#   def route(state) -> str:
#       return state["route"]
#   graph.add_conditional_edges("router", route, {"code": "code", "explain": "explain"})
#
# Command style (this example):
#   def router(state) -> Command:
#       route = classify(state["task"])
#       return Command(goto=route, update={"route": route})
#   # No add_conditional_edges needed!
#
# Command advantages:
#   - Routing logic co-located with decision-making code
#   - State update happens in the same return as routing
#   - Dynamic: goto can be any string computed at runtime
#   - Cleaner for complex routing with many conditions

from langgraph.types import Command

print(f"Command type: {type(Command)}")
print("Usage: return Command(goto='node_name', update={'key': 'value'})")

In [ ]:
# ── 4. Build the Command-pattern graph ────────────────────────────────────────
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.types import Command

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

ROUTER_PROMPT = """Classify this task into exactly one category: code, explain, math, or creative.
Task: {task}
Respond with only the category name."""

PROMPTS = {
    "code": "You are an expert programmer. Write clean, working code for: {task}",
    "explain": "You are a great teacher. Explain clearly in 3-5 sentences: {task}",
    "math": "You are a math tutor. Solve step by step: {task}",
    "creative": "You are a creative writer. Respond with creativity and humor: {task}",
}
VALID_ROUTES = set(PROMPTS.keys())


class CommandState(TypedDict):
    task: str
    route: str
    result: str


def router(state: CommandState) -> Command:
    result = llm.invoke([HumanMessage(content=ROUTER_PROMPT.format(task=state["task"]))])
    route = result.content.strip().lower()
    if route not in VALID_ROUTES:
        route = "explain"  # fallback for unrecognized categories
    print(f"  Router: '{state['task'][:40]}' → {route}")
    return Command(goto=route, update={"route": route})


def make_handler(category: str):
    def handler(state: CommandState) -> dict:
        prompt = PROMPTS[category].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"result": result.content}

    handler.__name__ = f"{category}_handler"
    return handler


graph = StateGraph(CommandState)
graph.add_node("router", router)
for category in VALID_ROUTES:
    graph.add_node(category, make_handler(category))
    graph.add_edge(category, END)
graph.set_entry_point("router")
# No add_conditional_edges -- Command handles routing
app = graph.compile()
print("Graph compiled.")

In [ ]:
# ── 5. Route 4 different task types ──────────────────────────────────────────
TASKS = [
    "Write a Python function to calculate fibonacci numbers.",
    "Explain what a REST API is in simple terms.",
    "What is 42 * 17? Show your work.",
    "Tell me a short joke about programming.",
]

for task in TASKS:
    print(f"\nTASK: {task}")
    result = app.invoke({"task": task, "route": "", "result": ""})
    print(f"Route: {result['route']}")
    print(f"Result: {result['result'][:200]}")

## Exercises

**Exercise 1 — Add interrupt:** Before each handler runs, add `interrupt()` to ask for user approval. This combines Command routing with HITL (see `11-hitl-approval`).

**Exercise 2 — Multi-hop routing:** After `code_handler`, add a `review` node that checks code quality. Use `Command(goto="review")` from the code handler to chain two hops.

**Exercise 3 — Command with resume:** Research `interrupt()` + `Command(resume=value)` for human-in-the-loop interrupts. When would you use `Command(resume=)` vs `Command(goto=)`?

**Exercise 4 — Confidence routing:** Have the router also output a confidence score. If confidence < 0.7, route to a fallback handler that asks for clarification before proceeding.

## What's next

- **11-hitl-approval** — `interrupt()` + `Command(resume=)` for human approval gates
- **39-checkpoint-resume** — `SqliteSaver` to persist graph state across Python sessions
- **6-multi-agent-graph** — `Command(goto=...)` across agent boundaries